In [5]:
!pip install tensorflow_datasets


  Using cached tensorflow_datasets-4.9.9-py3-none-any.whl.metadata (11 kB)
  Using cached immutabledict-4.3.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached promise-2.3.tar.gz (19 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached simple_parsing-0.1.8-py3-none-any.whl.metadata (8.1 kB)
  Using cached tensorflow_metadata-1.17.3-py3-none-any.whl.metadata (2.5 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
Using cached tensorflow_datasets-4.9.9-py3-none-any.whl (5.3 MB)
Using cached immutabledict-4.3.1

  You can safely remove it manually.


In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
# import seaborn as sns
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# import itertools

print(tf.__version__)

2.21.0


In [2]:
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

In [3]:
# Shuffle the training data first
indices = tf.random.shuffle(tf.range(len(x_train)))
x_train_shuffled = tf.gather(x_train, indices)
y_train_shuffled = tf.gather(y_train, indices)

# Split into training and validation (80/20)
val_split = 0.2
num_train = int(len(x_train_shuffled) * (1 - val_split))
x_train_final = x_train_shuffled[:num_train]
y_train_final = y_train_shuffled[:num_train]
x_val = x_train_shuffled[num_train:]
y_val = y_train_shuffled[num_train:]

print(f"Training set size: {len(x_train_final)}")
print(f"Validation set size: {len(x_val)}")
print(f"Test set size: {len(x_test)}")

Training set size: 40000
Validation set size: 10000
Test set size: 10000


In [4]:
dataset, info = tfds.load("cifar10", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples # 3670
class_names = info.features["label"].names # ["dandelion", "daisy", ...]
n_classes = info.features["label"].num_classes # 5

In [5]:
model = tf.keras.applications.xception.Xception(weights="imagenet")

In [6]:
sample_images = (x_train[:4].astype('float32')) / 255.0  # Normalize to 0-1
# Resize to 299x299 for Xception
images_resized = tf.image.resize(sample_images, (299, 299))

In [7]:
inputs = tf.keras.applications.xception.preprocess_input(images_resized)

In [8]:
batch_size = 32
preprocess = tf.keras.Sequential([
tf.keras.layers.Resizing(height=299, width=299, crop_to_aspect_ratio=True),
tf.keras.layers.Lambda(tf.keras.applications.xception.preprocess_input)
])

train_set = tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
valid_set = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
test_set = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)

In [9]:
data_augmentation = tf.keras.Sequential([
tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
tf.keras.layers.RandomRotation(factor=0.05, seed=42),
tf.keras.layers.RandomContrast(factor=0.2, seed=42)
])


In [10]:
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(n_classes, activation="softmax")(avg)
Model = tf.keras.Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=3)

Epoch 1/3
   1/1250 ━━━━━━━━━━━━━━━━━━━━ 26:58:29 78s/step - accuracy: 0.0000e+00 - loss: 7.6481